In [ ]:
%%capture
!pip install unsloth
!pip install --upgrade trl datasets huggingface_hub

In [ ]:
# ── Config ────────────────────────────────────────────────────────────────────
# Verify model ID at https://huggingface.co/unsloth before running
MODEL_ID      = "unsloth/gemma-4-e2b-it-bnb-4bit"
DATASET_PATH  = "/kaggle/input/sakhi-asha-sft/combined-asha-sft.jsonl"
OUTPUT_DIR    = "sakhi-gemma4-e2b-lora"
HF_REPO       = "docvm/sakhi-gemma4-e2b-asha-lora"
MAX_SEQ_LEN   = 512

In [ ]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

login(UserSecretsClient().get_secret("HF_TOKEN"))

In [ ]:
from unsloth import FastModel

model, tokenizer = FastModel.from_pretrained(
    model_name=MODEL_ID,
    max_seq_length=MAX_SEQ_LEN,
    load_in_4bit=True,
)

model = FastModel.get_peft_model(
    model,
    r=16,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    task_type="CAUSAL_LM",
)

model.print_trainable_parameters()

In [ ]:
from datasets import load_dataset

raw = load_dataset("json", data_files=DATASET_PATH, split="train")
train_ds = raw.filter(lambda x: x["split"] == "train")
eval_ds  = raw.filter(lambda x: x["split"] == "eval")

print(f"Train: {len(train_ds)} | Eval: {len(eval_ds)}")
print("\nSample entry:")
print(train_ds[0]["text"][:400])

In [ ]:
import torch
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LEN,
    dataset_num_proc=2,
    args=SFTConfig(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        num_train_epochs=3,
        warmup_steps=10,
        learning_rate=2e-4,
        bf16=torch.cuda.is_bf16_supported(),
        fp16=not torch.cuda.is_bf16_supported(),
        logging_steps=10,
        eval_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="cosine",
        output_dir=OUTPUT_DIR,
        report_to="none",
    ),
)

trainer.train()

In [ ]:
# Quick sanity check before pushing
FastModel.for_inference(model)

prompt = (
    "<start_of_turn>user\n"
    "I visited a pregnant woman at 32 weeks. She has a headache, swollen feet, "
    "and blurred vision. BP check not available. What should I do?\n"
    "<end_of_turn>\n"
    "<start_of_turn>model\n"
)

inputs = tokenizer(text=prompt, return_tensors="pt").to("cuda")
outputs = model.generate(**inputs, max_new_tokens=300, temperature=0.2, do_sample=True)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

In [ ]:
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

model.push_to_hub(HF_REPO, private=True)
tokenizer.push_to_hub(HF_REPO, private=True)

print(f"Adapter pushed to https://huggingface.co/{HF_REPO}")

In [ ]:
from huggingface_hub import HfApi
import datetime

model_card = """\
---
base_model: google/gemma-4-e2b-it
tags:
  - gemma
  - gemma-4
  - lora
  - qlora
  - unsloth
  - asha-worker
  - clinical-decision-support
  - maternal-health
  - newborn-health
  - india-healthcare
  - hindi
  - hinglish
  - multilingual
license: gemma
language:
  - en
  - hi
datasets:
  - docvm/sakhi-asha-sft
---

# Sakhi — Gemma 4 E2B ASHA LoRA

A LoRA adapter fine-tuned on **ASHA worker clinical guidelines** from India's MOHFW training modules.
Designed for [Sakhi](https://github.com/orcus108/sakhi) — a mobile-first clinical decision-support assistant for
Accredited Social Health Activists (ASHAs) in rural India.

## What it does

Given a field scenario described by an ASHA worker (patient observation, symptom report, or community health question),
the model responds with:

- **Risk Level** — High / Mid / Low / N/A
- **Assessment** — what the situation likely indicates
- **What to do** — concrete, step-by-step actions
- **Refer to PHC?** — clear referral decision

Responses are grounded in MOHFW/WHO guidelines and are available in English, Hindi, and Hinglish.

## Training

| Detail | Value |
|---|---|
| Base model | `google/gemma-4-e2b-it` |
| Method | QLoRA (4-bit NF4) via Unsloth |
| LoRA rank | 16 |
| LoRA alpha | 16 |
| Target modules | q, k, v, o, gate, up, down proj |
| Epochs | 3 |
| Learning rate | 2e-4 (cosine decay) |
| Batch size | 2 × grad_accum 4 = effective 8 |
| Max seq length | 512 |
| Optimizer | adamw_8bit |
| Hardware | Kaggle T4 |
| Train date | {date} |

## Dataset

**225 entries** across 4 ASHA training modules (Books 1–4), each in EN + HI + Hinglish.

Topics covered: ASHA tasks, water safety, immunization, diarrhoea/ORS, breastfeeding, tuberculosis,
snakebite, antenatal care, HIV/AIDS, contraception, acute respiratory infection, anaemia in pregnancy,
newborn care, postnatal care, family planning, RTI/STI, malaria, wound care, dog bites.

Source: MOHFW ASHA Module Training Books (public health guidelines).

## Inference

```python
from unsloth import FastModel

model, tokenizer = FastModel.from_pretrained(
    model_name="docvm/sakhi-gemma4-e2b-asha-lora",
    max_seq_length=512,
    load_in_4bit=True,
)
FastModel.for_inference(model)

prompt = (
    "<start_of_turn>user\\n"
    "I visited a pregnant woman at 36 weeks. She has a severe headache and blurred vision.\\n"
    "<end_of_turn>\\n"
    "<start_of_turn>model\\n"
)

inputs = tokenizer(text=prompt, return_tensors="pt").to("cuda")
outputs = model.generate(**inputs, max_new_tokens=300, temperature=0.2, do_sample=True)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))
```

## Intended use

- Clinical decision support for ASHA workers via the Sakhi app
- ASHA training and simulation
- Research into multilingual community health AI

## Limitations and safety

- **Never diagnoses** — flags and supports referral decisions only
- All responses recommend PHC referral when uncertainty exists
- Hindi clinical accuracy is unverified against a formal benchmark — treat outputs as decision-support, not ground truth
- Must not be used for autonomous medical decision-making or direct-to-patient deployment without human oversight

## License

[Gemma Terms of Use](https://ai.google.dev/gemma/terms) apply.
""".format(date=datetime.date.today().isoformat())

api = HfApi()
api.upload_file(
    path_or_fileobj=model_card.encode(),
    path_in_repo="README.md",
    repo_id=HF_REPO,
    repo_type="model",
    commit_message="Add model card",
)

print(f"Model card updated → https://huggingface.co/{HF_REPO}")